<a href="https://colab.research.google.com/github/Iqra171/fairness-in-federated-learning/blob/main/RealIncomeDataset/adult_fairness_attacks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score
import random
import matplotlib.pyplot as plt
from collections import defaultdict
import pandas as pd
import os
import warnings

warnings.filterwarnings('ignore')

"""
================================================================================
EXPERIMENT 2: Adult Income Dataset (Real-World) - Multi-Seed Analysis
================================================================================
"""

# ==============================================================================
# CONFIGURATION
# ==============================================================================

class Config:
    """Configuration class for experiment parameters"""
    N_CLIENTS = 6
    N_ROUNDS = 20
    LOCAL_EPOCHS = 5
    LEARNING_RATE = 0.01
    ADVERSARY_INDICES = [4, 5]
    ATTACK_START_ROUND = 5
    SEED = 42

    # Output directory
    OUTPUT_DIR = "adult_experiment_results"

    @classmethod
    def set_seed(cls, seed=None):
        """Set random seeds for reproducibility"""
        if seed is not None:
            cls.SEED = seed
        random.seed(cls.SEED)
        np.random.seed(cls.SEED)
        torch.manual_seed(cls.SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(cls.SEED)

    @classmethod
    def create_output_dir(cls):
        """Create output directory if it doesn't exist"""
        if not os.path.exists(cls.OUTPUT_DIR):
            os.makedirs(cls.OUTPUT_DIR)
            print(f"[INFO] Created output directory: {cls.OUTPUT_DIR}")


# ==============================================================================
# DATA LOADING
# ==============================================================================

class DataLoader:
    """Class for loading and preprocessing the Adult Income dataset"""

    COLUMNS = [
        'age', 'workclass', 'fnlwgt', 'education', 'education_num',
        'marital_status', 'occupation', 'relationship', 'race', 'sex',
        'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
    ]

    TRAIN_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
    TEST_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"

    NUM_COLS = ['age', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week', 'fnlwgt']
    CAT_COLS = ['workclass', 'education', 'marital_status', 'occupation', 'relationship', 'race', 'native_country']

    @classmethod
    def load_adult_data(cls):
        """Load UCI Adult Income Dataset"""
        try:
            # Check for cached data
            if os.path.exists('adult_data.csv'):
                print("[DATA] Loading cached Adult dataset...")
                df = pd.read_csv('adult_data.csv')
            else:
                print("[DATA] Downloading Adult dataset...")
                df_train = pd.read_csv(cls.TRAIN_URL, names=cls.COLUMNS, skipinitialspace=True)
                df_test = pd.read_csv(cls.TEST_URL, names=cls.COLUMNS, skipinitialspace=True, skiprows=1)
                df = pd.concat([df_train, df_test], ignore_index=True)
                df.to_csv('adult_data.csv', index=False)
                print("[DATA] Dataset cached to adult_data.csv")

            # Clean data
            df = df.replace(' ?', np.nan).dropna()

            # Extract sensitive attribute (sex)
            sensitive = (df['sex'].str.strip() == 'Male').astype(int).values

            # Extract target (income)
            df['income'] = df['income'].str.strip().str.rstrip('.')
            y = (df['income'] == '>50K').astype(int).values

            # Extract and process features
            X_num = df[cls.NUM_COLS].values.astype(float)

            X_cat = []
            for col in cls.CAT_COLS:
                le = LabelEncoder()
                X_cat.append(le.fit_transform(df[col].str.strip()).reshape(-1, 1))
            X_cat = np.hstack(X_cat)

            X = np.hstack([X_num, X_cat])

            # Standardize features
            scaler = StandardScaler()
            X = scaler.fit_transform(X)

            print(f"[DATA] Adult Dataset loaded: {len(X)} samples, {X.shape[1]} features")
            print(f"[DATA] Positive rate: {y.mean():.3f}, Male rate: {sensitive.mean():.3f}")

            return X, y, sensitive

        except Exception as e:
            print(f"[ERROR] Failed to load Adult dataset: {e}")
            raise

    @classmethod
    def create_clients(cls, X, y, sensitive):
        """Create federated clients with heterogeneous data"""
        n = len(X)
        indices = np.arange(n)
        np.random.shuffle(indices)
        splits = np.array_split(indices, Config.N_CLIENTS)

        clients = []
        for i, split in enumerate(splits):
            local_bias = 0.05 + (i * 0.08)

            X_c = X[split].copy()
            y_c = y[split].copy()
            s_c = sensitive[split].copy()

            # Apply local bias
            for j in range(len(y_c)):
                if s_c[j] == 1 and random.random() < local_bias:
                    y_c[j] = 1 - y_c[j]

            clients.append({
                'id': i,
                'X': X_c,
                'y': y_c,
                'sensitive': s_c,
                'bias_rate': local_bias
            })

        return clients


# ==============================================================================
# MODEL DEFINITION
# ==============================================================================

class FederatedModel(nn.Module):
    """Neural network model for federated learning"""

    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

    def predict_proba(self, x):
        """Get probability predictions"""
        with torch.no_grad():
            return torch.sigmoid(self.forward(x))


# ==============================================================================
# FAIRNESS METRICS
# ==============================================================================

class FairnessMetrics:
    """Class for computing fairness metrics"""

    @staticmethod
    def demographic_parity(preds, sensitive):
        """Compute demographic parity gap"""
        preds = np.asarray(preds).flatten()
        binary = (preds > 0.5).astype(int)

        r0 = binary[sensitive == 0].mean() if np.any(sensitive == 0) else 0.5
        r1 = binary[sensitive == 1].mean() if np.any(sensitive == 1) else 0.5

        return abs(r0 - r1)

    @staticmethod
    def equalized_odds(preds, labels, sensitive):
        """Compute equalized odds gap"""
        preds = np.asarray(preds).flatten()
        binary = (preds > 0.5).astype(int)

        m0 = (sensitive == 0) & (labels == 1)
        m1 = (sensitive == 1) & (labels == 1)

        tpr0 = binary[m0].mean() if np.any(m0) else 0.5
        tpr1 = binary[m1].mean() if np.any(m1) else 0.5

        return abs(tpr0 - tpr1)


# ==============================================================================
# TRAINING UTILITIES
# ==============================================================================

class TrainingUtils:
    """Utility functions for model training"""

    @staticmethod
    def safe_train(model, X, y, epochs=5, lr=0.01, batch_size=256):
        """Safe mini-batch training with BCEWithLogitsLoss"""
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
        loss_fn = nn.BCEWithLogitsLoss()

        X_t = torch.tensor(X, dtype=torch.float32)
        y_t = torch.tensor(y, dtype=torch.float32).view(-1, 1)

        n_samples = len(X)

        model.train()
        for _ in range(epochs):
            indices = np.random.permutation(n_samples)

            for start in range(0, n_samples, batch_size):
                end = min(start + batch_size, n_samples)
                batch_idx = indices[start:end]

                optimizer.zero_grad()
                logits = model(X_t[batch_idx])
                loss = loss_fn(logits, y_t[batch_idx])

                if torch.isnan(loss):
                    continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        return model.state_dict()

    @staticmethod
    def honest_train(global_weights, client, input_dim):
        """Honest client training"""
        model = FederatedModel(input_dim=input_dim)
        model.load_state_dict(global_weights)
        return TrainingUtils.safe_train(
            model, client['X'], client['y'],
            Config.LOCAL_EPOCHS, Config.LEARNING_RATE
        )


# ==============================================================================
# ATTACK IMPLEMENTATIONS
# ==============================================================================

class Attacks:
    """Collection of attack strategies"""

    @staticmethod
    def none(global_weights, client, input_dim):
        """No attack - honest training"""
        return TrainingUtils.honest_train(global_weights, client, input_dim)

    @staticmethod
    def random_weights(global_weights, client, input_dim):
        """Random weight attack"""
        return FederatedModel(input_dim=input_dim).state_dict()

    @staticmethod
    def fairness_poison(global_weights, client, input_dim):
        """Fairness poisoning attack"""
        model = FederatedModel(input_dim=input_dim)
        model.load_state_dict(global_weights)

        # Adversarial labels: 0 for group 1
        adv_y = np.zeros(len(client['y']))
        adv_y[client['sensitive'] == 0] = client['y'][client['sensitive'] == 0]

        return TrainingUtils.safe_train(model, client['X'], adv_y, epochs=10, lr=0.03)

    @staticmethod
    def label_flip(global_weights, client, input_dim):
        """Label flipping attack"""
        model = FederatedModel(input_dim=input_dim)
        model.load_state_dict(global_weights)

        flipped = client['y'].copy()
        for i in range(len(flipped)):
            if client['sensitive'][i] == 1 and random.random() < 0.5:
                flipped[i] = 1 - flipped[i]

        return TrainingUtils.safe_train(
            model, client['X'], flipped,
            Config.LOCAL_EPOCHS, Config.LEARNING_RATE
        )

    @staticmethod
    def gradient_scale(global_weights, client, input_dim):
        """Gradient scaling attack"""
        honest = TrainingUtils.honest_train(global_weights, client, input_dim)

        scaled = {}
        for key in honest.keys():
            diff = honest[key] - global_weights[key]
            scaled[key] = global_weights[key] + 1.5 * torch.clamp(diff, -0.3, 0.3)

        return scaled

    @staticmethod
    def fairness_spoof(global_weights, client, input_dim):
        """Fairness spoofing attack"""
        model = FederatedModel(input_dim=input_dim)
        model.load_state_dict(global_weights)

        optimizer = optim.Adam(model.parameters(), lr=0.003, weight_decay=1e-4)
        loss_fn = nn.BCEWithLogitsLoss()

        # Use subset for faster training
        n = min(1000, len(client['X']))
        idx = np.random.choice(len(client['X']), n, replace=False)

        X = torch.tensor(client['X'][idx], dtype=torch.float32)
        y = torch.tensor(client['y'][idx], dtype=torch.float32).view(-1, 1)
        sens = client['sensitive'][idx]

        mask_0 = torch.tensor(sens == 0)
        mask_1 = torch.tensor(sens == 1)

        model.train()
        for _ in range(10):
            optimizer.zero_grad()

            logits = model(X)
            probs = torch.sigmoid(logits)

            ce_loss = loss_fn(logits, y)

            # Local fairness matching
            if mask_0.any() and mask_1.any():
                fair_loss = (probs[mask_0].mean() - probs[mask_1].mean()) ** 2
            else:
                fair_loss = torch.tensor(0.0)

            # Hidden bias
            if mask_1.any():
                bias = 0.03 * probs[mask_1].mean()
            else:
                bias = torch.tensor(0.0)

            total = ce_loss + 0.1 * fair_loss + bias

            if torch.isnan(total):
                break

            total.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            optimizer.step()

        return model.state_dict()

    @staticmethod
    def adaptive(global_weights, client, input_dim):
        """Adaptive attack designed to evade detection"""
        model = FederatedModel(input_dim=input_dim)
        model.load_state_dict(global_weights)

        optimizer = optim.Adam(model.parameters(), lr=0.002, weight_decay=1e-4)
        loss_fn = nn.BCEWithLogitsLoss()

        n = min(1000, len(client['X']))
        idx = np.random.choice(len(client['X']), n, replace=False)

        X = torch.tensor(client['X'][idx], dtype=torch.float32)
        y = torch.tensor(client['y'][idx], dtype=torch.float32).view(-1, 1)
        sens = client['sensitive'][idx]

        mask_0 = torch.tensor(sens == 0)
        mask_1 = torch.tensor(sens == 1)

        model.train()
        for _ in range(15):
            optimizer.zero_grad()

            logits = model(X)
            probs = torch.sigmoid(logits)

            ce_loss = loss_fn(logits, y)

            # Stay fair locally
            if mask_0.any() and mask_1.any():
                fair_loss = torch.abs(probs[mask_0].mean() - probs[mask_1].mean())
            else:
                fair_loss = torch.tensor(0.0)

            # Subtle bias
            if mask_1.any():
                bias = -0.01 * ((probs[mask_1] - 0.5) ** 2).mean()
            else:
                bias = torch.tensor(0.0)

            total = ce_loss + 0.2 * fair_loss + bias

            if torch.isnan(total):
                break

            total.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.3)
            optimizer.step()

        return model.state_dict()


# ==============================================================================
# AGGREGATION METHODS (DEFENSES)
# ==============================================================================

class Aggregators:
    """Collection of aggregation/defense strategies"""

    @staticmethod
    def fedavg(weights_list, **kwargs):
        """Federated Averaging"""
        agg = {}
        for key in weights_list[0].keys():
            stacked = torch.stack([w[key].float() for w in weights_list])
            agg[key] = torch.mean(stacked, dim=0)
        return agg, []

    @staticmethod
    def median(weights_list, **kwargs):
        """Coordinate-wise median aggregation"""
        agg = {}
        for key in weights_list[0].keys():
            stacked = torch.stack([w[key].float() for w in weights_list])
            agg[key] = torch.median(stacked, dim=0)[0]
        return agg, []

    @staticmethod
    def trimmed_mean(weights_list, trim=1, **kwargs):
        """Trimmed mean aggregation"""
        agg = {}
        for key in weights_list[0].keys():
            stacked = torch.stack([w[key].float() for w in weights_list])
            sorted_w, _ = torch.sort(stacked, dim=0)
            if len(weights_list) > 2 * trim:
                trimmed = sorted_w[trim:-trim]
            else:
                trimmed = sorted_w
            agg[key] = torch.mean(trimmed, dim=0)
        return agg, []

    @staticmethod
    def krum(weights_list, n_bad=2, **kwargs):
        """Krum aggregation"""
        n = len(weights_list)
        n_select = max(1, n - n_bad)

        # Compute pairwise distances
        dists = np.zeros((n, n))
        for i in range(n):
            for j in range(i + 1, n):
                d = sum(
                    torch.norm(weights_list[i][k].float() - weights_list[j][k].float()).item() ** 2
                    for k in weights_list[i].keys()
                )
                dists[i, j] = dists[j, i] = np.sqrt(d)

        # Compute scores
        scores = [np.sum(np.sort(dists[i])[1:n_select]) for i in range(n)]
        selected = np.argsort(scores)[:n_select]

        # Aggregate selected
        agg = {}
        for key in weights_list[0].keys():
            stacked = torch.stack([weights_list[i][key].float() for i in selected])
            agg[key] = torch.mean(stacked, dim=0)

        excluded = [i for i in range(n) if i not in selected]
        return agg, excluded

    @staticmethod
    def fairness_only(weights_list, clients, input_dim, **kwargs):
        """Fairness-weighted aggregation"""
        scores = []

        for w, c in zip(weights_list, clients):
            model = FederatedModel(input_dim=input_dim)
            model.load_state_dict(w)
            model.eval()

            # Use subset for speed
            n = min(500, len(c['X']))
            idx = np.random.choice(len(c['X']), n, replace=False)

            with torch.no_grad():
                preds = model.predict_proba(
                    torch.tensor(c['X'][idx], dtype=torch.float32)
                ).numpy()

            dp = FairnessMetrics.demographic_parity(preds, c['sensitive'][idx])
            scores.append(dp)

        # Lower DP gap = higher weight
        weights = 1.0 / (np.array(scores) + 0.01)
        weights = weights / weights.sum()

        agg = {}
        for key in weights_list[0].keys():
            weighted = torch.zeros_like(weights_list[0][key].float())
            for i, w in enumerate(weights_list):
                weighted += weights[i] * w[key].float()
            agg[key] = weighted

        return agg, []

    @staticmethod
    def robust_fair(weights_list, clients, global_weights, X_all, s_all, input_dim, **kwargs):
        """Robust fairness-aware aggregation"""
        n = len(weights_list)

        # Compute distance scores
        distances = []
        for w in weights_list:
            d = sum(
                torch.norm(w[k].float() - global_weights[k].float()).item() ** 2
                for k in global_weights.keys()
            )
            distances.append(np.sqrt(d))

        distances = np.array(distances)
        z_scores = (distances - distances.mean()) / (distances.std() + 1e-6)

        # Compute fairness consistency (use subset)
        n_eval = min(500, len(X_all))
        eval_idx = np.random.choice(len(X_all), n_eval, replace=False)

        consistency = []
        for w, c in zip(weights_list, clients):
            model = FederatedModel(input_dim=input_dim)
            model.load_state_dict(w)
            model.eval()

            n_local = min(500, len(c['X']))
            local_idx = np.random.choice(len(c['X']), n_local, replace=False)

            with torch.no_grad():
                local_p = model.predict_proba(
                    torch.tensor(c['X'][local_idx], dtype=torch.float32)
                ).numpy()
                global_p = model.predict_proba(
                    torch.tensor(X_all[eval_idx], dtype=torch.float32)
                ).numpy()

            local_dp = FairnessMetrics.demographic_parity(local_p, c['sensitive'][local_idx])
            global_dp = FairnessMetrics.demographic_parity(global_p, s_all[eval_idx])
            consistency.append(abs(local_dp - global_dp))

        consistency = np.array(consistency)

        # Compute trust scores
        trust = []
        suspicious = []
        for i in range(n):
            t = 1.0 / (1.0 + abs(z_scores[i]) + consistency[i])
            if z_scores[i] > 2.0 or consistency[i] > 0.25:
                suspicious.append(i)
                t *= 0.1
            trust.append(t)

        trust = np.array(trust)
        trust = trust / trust.sum()

        # Weighted aggregation
        agg = {}
        for key in weights_list[0].keys():
            weighted = torch.zeros_like(weights_list[0][key].float())
            for i, w in enumerate(weights_list):
                weighted += trust[i] * w[key].float()
            agg[key] = weighted

        return agg, suspicious


# ==============================================================================
# EXPERIMENT RUNNER
# ==============================================================================

class ExperimentRunner:
    """Class for running federated learning experiments"""

    # Attack function mapping
    ATTACKS = {
        'none': Attacks.none,
        'random': Attacks.random_weights,
        'fairness_poison': Attacks.fairness_poison,
        'label_flip': Attacks.label_flip,
        'gradient_scale': Attacks.gradient_scale,
        'fairness_spoof': Attacks.fairness_spoof,
        'adaptive': Attacks.adaptive
    }

    # Defense function mapping
    DEFENSES = {
        'fedavg': Aggregators.fedavg,
        'median': Aggregators.median,
        'trimmed': Aggregators.trimmed_mean,
        'krum': Aggregators.krum,
        'fairness_only': Aggregators.fairness_only,
        'robust_fair': Aggregators.robust_fair
    }

    @staticmethod
    def run_single_experiment(attack_name, defense_name, X, y, sensitive, clients, input_dim):
        """Run a single experiment configuration"""
        attack_fn = ExperimentRunner.ATTACKS[attack_name]
        agg_fn = ExperimentRunner.DEFENSES[defense_name]

        global_model = FederatedModel(input_dim=input_dim)
        history = defaultdict(list)

        # Evaluation subset for efficiency
        eval_n = min(2000, len(X))
        eval_idx = np.random.choice(len(X), eval_n, replace=False)

        for round_num in range(Config.N_ROUNDS):
            attack_active = round_num >= Config.ATTACK_START_ROUND
            client_weights = []

            # Local training
            for i, client in enumerate(clients):
                is_adv = i in Config.ADVERSARY_INDICES and attack_active

                try:
                    if is_adv:
                        w = attack_fn(global_model.state_dict(), client, input_dim)
                    else:
                        w = TrainingUtils.honest_train(
                            global_model.state_dict(), client, input_dim
                        )
                except Exception:
                    w = TrainingUtils.honest_train(
                        global_model.state_dict(), client, input_dim
                    )

                client_weights.append(w)

            # Aggregation
            try:
                agg, _ = agg_fn(
                    client_weights,
                    clients=clients,
                    global_weights=global_model.state_dict(),
                    X_all=X,
                    s_all=sensitive,
                    input_dim=input_dim
                )
                global_model.load_state_dict(agg)
            except Exception:
                agg, _ = Aggregators.fedavg(client_weights)
                global_model.load_state_dict(agg)

            # Evaluation
            global_model.eval()
            with torch.no_grad():
                preds = global_model.predict_proba(
                    torch.tensor(X[eval_idx], dtype=torch.float32)
                ).numpy()

            acc = accuracy_score(y[eval_idx], (preds > 0.5).astype(int))
            dp = FairnessMetrics.demographic_parity(preds, sensitive[eval_idx])
            eo = FairnessMetrics.equalized_odds(preds, y[eval_idx], sensitive[eval_idx])

            history['accuracy'].append(acc)
            history['demographic_parity'].append(dp)
            history['equalized_odds'].append(eo)
            history['round'].append(round_num + 1)

        return dict(history)


# ==============================================================================
# VISUALIZATION
# ==============================================================================

class Visualizer:
    """Class for generating experiment visualizations"""

    # Color schemes
    ATTACK_COLORS = {
        'none': 'green',
        'random': 'orange',
        'fairness_poison': 'red',
        'label_flip': 'purple',
        'gradient_scale': 'blue',
        'fairness_spoof': 'magenta',
        'adaptive': 'black'
    }

    DEFENSE_MARKERS = {
        'fedavg': 'o',
        'median': 's',
        'trimmed': 'd',
        'krum': 'p',
        'fairness_only': '^',
        'robust_fair': '*'
    }

    DEFENSE_COLORS = {
        'fedavg': '#1f77b4',
        'median': '#ff7f0e',
        'trimmed': '#2ca02c',
        'krum': '#d62728',
        'fairness_only': '#9467bd',
        'robust_fair': '#8c564b'
    }

    @staticmethod
    def plot_single_seed_results(results, seed, save_dir):
        """Generate comprehensive plots for a single seed"""
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        fig.suptitle(f'Adult Income Dataset - Federated Learning Results (Seed {seed})',
                     fontsize=16, fontweight='bold')

        rounds = list(range(1, Config.N_ROUNDS + 1))

        # Plot 1: Accuracy over rounds (attacks vs fedavg)
        ax = axes[0, 0]
        attacks = ['none', 'random', 'fairness_poison', 'fairness_spoof', 'adaptive']

        for attack in attacks:
            key = (attack, 'fedavg')
            if key in results:
                color = Visualizer.ATTACK_COLORS.get(attack, 'gray')
                ax.plot(rounds, results[key]['accuracy'],
                       label=attack, color=color, linewidth=2, marker='o', markersize=4)

        ax.axvline(x=Config.ATTACK_START_ROUND + 1, color='gray', linestyle='--',
                   alpha=0.7, label='Attack Start')
        ax.set_xlabel('Round', fontsize=11)
        ax.set_ylabel('Accuracy', fontsize=11)
        ax.set_title('Accuracy Under Different Attacks (vs FedAvg)', fontsize=12)
        ax.legend(loc='lower left', fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.set_ylim([0.5, 0.95])

        # Plot 2: Demographic Parity over rounds (attacks vs fedavg)
        ax = axes[0, 1]

        for attack in attacks:
            key = (attack, 'fedavg')
            if key in results:
                color = Visualizer.ATTACK_COLORS.get(attack, 'gray')
                ax.plot(rounds, results[key]['demographic_parity'],
                       label=attack, color=color, linewidth=2, marker='o', markersize=4)

        ax.axvline(x=Config.ATTACK_START_ROUND + 1, color='gray', linestyle='--', alpha=0.7)
        ax.set_xlabel('Round', fontsize=11)
        ax.set_ylabel('Demographic Parity Gap', fontsize=11)
        ax.set_title('Fairness Under Different Attacks (vs FedAvg)', fontsize=12)
        ax.legend(loc='upper left', fontsize=9)
        ax.grid(True, alpha=0.3)

        # Plot 3: Equalized Odds over rounds
        ax = axes[0, 2]

        for attack in attacks:
            key = (attack, 'fedavg')
            if key in results:
                color = Visualizer.ATTACK_COLORS.get(attack, 'gray')
                ax.plot(rounds, results[key]['equalized_odds'],
                       label=attack, color=color, linewidth=2, marker='o', markersize=4)

        ax.axvline(x=Config.ATTACK_START_ROUND + 1, color='gray', linestyle='--', alpha=0.7)
        ax.set_xlabel('Round', fontsize=11)
        ax.set_ylabel('Equalized Odds Gap', fontsize=11)
        ax.set_title('Equalized Odds Under Different Attacks', fontsize=12)
        ax.legend(loc='upper left', fontsize=9)
        ax.grid(True, alpha=0.3)

        # Plot 4: Defense comparison vs fairness_poison
        ax = axes[1, 0]
        defenses = ['fedavg', 'median', 'robust_fair']

        for defense in defenses:
            key = ('fairness_poison', defense)
            if key in results:
                color = Visualizer.DEFENSE_COLORS.get(defense, 'gray')
                ls = '--' if defense == 'fedavg' else '-'
                lw = 3 if defense in ['robust_fair', 'fedavg'] else 2
                ax.plot(rounds, results[key]['demographic_parity'],
                       label=defense, color=color, linestyle=ls, linewidth=lw,
                       marker='s', markersize=4)

        ax.axvline(x=Config.ATTACK_START_ROUND + 1, color='gray', linestyle='--', alpha=0.7)
        ax.set_xlabel('Round', fontsize=11)
        ax.set_ylabel('Demographic Parity Gap', fontsize=11)
        ax.set_title('Defense Effectiveness vs Fairness Poisoning', fontsize=12)
        ax.legend(loc='upper left', fontsize=9)
        ax.grid(True, alpha=0.3)

        # Plot 5: Defense comparison vs adaptive attack
        ax = axes[1, 1]

        for defense in defenses:
            key = ('adaptive', defense)
            if key in results:
                color = Visualizer.DEFENSE_COLORS.get(defense, 'gray')
                ls = '--' if defense == 'fedavg' else '-'
                lw = 3 if defense in ['robust_fair', 'fedavg'] else 2
                ax.plot(rounds, results[key]['demographic_parity'],
                       label=defense, color=color, linestyle=ls, linewidth=lw,
                       marker='s', markersize=4)

        ax.axvline(x=Config.ATTACK_START_ROUND + 1, color='gray', linestyle='--', alpha=0.7)
        ax.set_xlabel('Round', fontsize=11)
        ax.set_ylabel('Demographic Parity Gap', fontsize=11)
        ax.set_title('Defense Effectiveness vs Adaptive Attack', fontsize=12)
        ax.legend(loc='upper left', fontsize=9)
        ax.grid(True, alpha=0.3)

        # Plot 6: Accuracy vs Fairness tradeoff (scatter)
        ax = axes[1, 2]

        for (attack, defense), history in results.items():
            color = Visualizer.ATTACK_COLORS.get(attack, 'gray')
            marker = Visualizer.DEFENSE_MARKERS.get(defense, 'o')

            ax.scatter(
                history['accuracy'][-1],
                history['demographic_parity'][-1],
                c=color, marker=marker, s=150, alpha=0.7,
                edgecolors='black', linewidth=0.5,
                label=f"{attack[:6]}/{defense[:6]}"
            )

        ax.set_xlabel('Final Accuracy', fontsize=11)
        ax.set_ylabel('Final DP Gap', fontsize=11)
        ax.set_title('Accuracy-Fairness Tradeoff', fontsize=12)
        ax.grid(True, alpha=0.3)

        # Add legend explanation
        ax.text(0.02, 0.98, 'Colors = Attacks\nMarkers = Defenses',
               transform=ax.transAxes, fontsize=9, verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

        plt.tight_layout(rect=[0, 0.03, 1, 0.95])

        # Save figure
        filename = os.path.join(save_dir, f'adult_results_seed_{seed}.png')
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"[SAVED] {filename}")
        plt.close()

    @staticmethod
    def plot_comparison_across_seeds(all_runs, seeds, save_dir):
        """Generate comparison plots across all seeds"""

        # Collect metrics for each experiment across seeds
        metrics_by_exp = defaultdict(lambda: {'accuracy': [], 'dp': [], 'eo': []})

        for run_results in all_runs:
            for (attack, defense), history in run_results.items():
                key = (attack, defense)
                metrics_by_exp[key]['accuracy'].append(history['accuracy'][-1])
                metrics_by_exp[key]['dp'].append(history['demographic_parity'][-1])
                metrics_by_exp[key]['eo'].append(history['equalized_odds'][-1])

        # Create comparison figure
        fig, axes = plt.subplots(2, 2, figsize=(16, 14))
        fig.suptitle(f'Adult Dataset - Multi-Seed Comparison (Seeds: {seeds})',
                     fontsize=16, fontweight='bold')

        # Plot 1: Accuracy comparison bar chart
        ax = axes[0, 0]
        experiments = list(metrics_by_exp.keys())
        x = np.arange(len(experiments))

        means = [np.mean(metrics_by_exp[exp]['accuracy']) for exp in experiments]
        stds = [np.std(metrics_by_exp[exp]['accuracy']) for exp in experiments]

        colors = [Visualizer.ATTACK_COLORS.get(exp[0], 'gray') for exp in experiments]
        bars = ax.bar(x, means, yerr=stds, capsize=3, color=colors, alpha=0.7, edgecolor='black')

        ax.set_xlabel('Experiment', fontsize=11)
        ax.set_ylabel('Accuracy', fontsize=11)
        ax.set_title('Final Accuracy (Mean ± Std across Seeds)', fontsize=12)
        ax.set_xticks(x)
        ax.set_xticklabels([f"{a[:4]}/{d[:4]}" for a, d in experiments],
                          rotation=45, ha='right', fontsize=8)
        ax.grid(True, alpha=0.3, axis='y')

        # Plot 2: DP comparison bar chart
        ax = axes[0, 1]

        means = [np.mean(metrics_by_exp[exp]['dp']) for exp in experiments]
        stds = [np.std(metrics_by_exp[exp]['dp']) for exp in experiments]

        bars = ax.bar(x, means, yerr=stds, capsize=3, color=colors, alpha=0.7, edgecolor='black')

        ax.set_xlabel('Experiment', fontsize=11)
        ax.set_ylabel('Demographic Parity Gap', fontsize=11)
        ax.set_title('Final DP Gap (Mean ± Std across Seeds)', fontsize=12)
        ax.set_xticks(x)
        ax.set_xticklabels([f"{a[:4]}/{d[:4]}" for a, d in experiments],
                          rotation=45, ha='right', fontsize=8)
        ax.grid(True, alpha=0.3, axis='y')

        # Plot 3: Box plot for accuracy by attack type
        ax = axes[1, 0]

        attack_types = ['none', 'fairness_poison', 'fairness_spoof', 'adaptive']
        defense = 'fedavg'

        box_data = []
        labels = []
        box_colors = []
        for attack in attack_types:
            key = (attack, defense)
            if key in metrics_by_exp:
                box_data.append(metrics_by_exp[key]['accuracy'])
                labels.append(attack)
                box_colors.append(Visualizer.ATTACK_COLORS.get(attack, 'gray'))

        bp = ax.boxplot(box_data, labels=labels, patch_artist=True)
        for patch, color in zip(bp['boxes'], box_colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)

        ax.set_xlabel('Attack Type', fontsize=11)
        ax.set_ylabel('Accuracy', fontsize=11)
        ax.set_title('Accuracy Distribution by Attack (vs FedAvg)', fontsize=12)
        ax.grid(True, alpha=0.3, axis='y')

        # Plot 4: Box plot for DP gap by defense type
        ax = axes[1, 1]

        attack = 'fairness_poison'
        defense_types = ['fedavg', 'median', 'robust_fair']

        box_data = []
        labels = []
        for defense in defense_types:
            key = (attack, defense)
            if key in metrics_by_exp:
                box_data.append(metrics_by_exp[key]['dp'])
                labels.append(defense)

        bp = ax.boxplot(box_data, labels=labels, patch_artist=True)
        colors = [Visualizer.DEFENSE_COLORS.get(d, 'gray') for d in defense_types if (attack, d) in metrics_by_exp]
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)

        ax.set_xlabel('Defense Type', fontsize=11)
        ax.set_ylabel('Demographic Parity Gap', fontsize=11)
        ax.set_title('DP Gap Distribution by Defense (vs Fairness Poison)', fontsize=12)
        ax.grid(True, alpha=0.3, axis='y')

        plt.tight_layout(rect=[0, 0.03, 1, 0.95])

        # Save figure
        filename = os.path.join(save_dir, 'adult_comparison_across_seeds.png')
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"[SAVED] {filename}")
        plt.close()

    @staticmethod
    def plot_time_series_comparison(all_runs, seeds, save_dir):
        """Plot time series with confidence bands across seeds"""

        # Collect time series data
        time_series = defaultdict(lambda: defaultdict(list))

        for run_results in all_runs:
            for (attack, defense), history in run_results.items():
                key = (attack, defense)
                time_series[key]['accuracy'].append(history['accuracy'])
                time_series[key]['dp'].append(history['demographic_parity'])

        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle(f'Adult Dataset - Time Series with Confidence Bands (Seeds: {seeds})',
                     fontsize=16, fontweight='bold')

        rounds = np.arange(1, Config.N_ROUNDS + 1)

        # Plot 1: Accuracy for different attacks
        ax = axes[0, 0]
        attacks = ['none', 'fairness_poison', 'fairness_spoof', 'adaptive']
        defense = 'fedavg'

        for attack in attacks:
            key = (attack, defense)
            if key in time_series:
                data = np.array(time_series[key]['accuracy'])
                mean = np.mean(data, axis=0)
                std = np.std(data, axis=0)

                color = Visualizer.ATTACK_COLORS.get(attack, 'gray')
                ax.plot(rounds, mean, label=attack, color=color, linewidth=2)
                ax.fill_between(rounds, mean - std, mean + std, color=color, alpha=0.2)

        ax.axvline(x=Config.ATTACK_START_ROUND + 1, color='gray', linestyle='--', alpha=0.7)
        ax.set_xlabel('Round', fontsize=11)
        ax.set_ylabel('Accuracy', fontsize=11)
        ax.set_title('Accuracy with Confidence Bands (Attacks vs FedAvg)', fontsize=12)
        ax.legend(loc='lower left', fontsize=9)
        ax.grid(True, alpha=0.3)

        # Plot 2: DP for different attacks
        ax = axes[0, 1]

        for attack in attacks:
            key = (attack, defense)
            if key in time_series:
                data = np.array(time_series[key]['dp'])
                mean = np.mean(data, axis=0)
                std = np.std(data, axis=0)

                color = Visualizer.ATTACK_COLORS.get(attack, 'gray')
                ax.plot(rounds, mean, label=attack, color=color, linewidth=2)
                ax.fill_between(rounds, mean - std, mean + std, color=color, alpha=0.2)

        ax.axvline(x=Config.ATTACK_START_ROUND + 1, color='gray', linestyle='--', alpha=0.7)
        ax.set_xlabel('Round', fontsize=11)
        ax.set_ylabel('Demographic Parity Gap', fontsize=11)
        ax.set_title('DP Gap with Confidence Bands (Attacks vs FedAvg)', fontsize=12)
        ax.legend(loc='upper left', fontsize=9)
        ax.grid(True, alpha=0.3)

        # Plot 3: Accuracy for different defenses
        ax = axes[1, 0]
        attack = 'fairness_poison'
        defenses = ['fedavg', 'median', 'robust_fair']

        for defense in defenses:
            key = (attack, defense)
            if key in time_series:
                data = np.array(time_series[key]['accuracy'])
                mean = np.mean(data, axis=0)
                std = np.std(data, axis=0)

                color = Visualizer.DEFENSE_COLORS.get(defense, 'gray')
                ax.plot(rounds, mean, label=defense, color=color, linewidth=2)
                ax.fill_between(rounds, mean - std, mean + std, color=color, alpha=0.2)

        ax.axvline(x=Config.ATTACK_START_ROUND + 1, color='gray', linestyle='--', alpha=0.7)
        ax.set_xlabel('Round', fontsize=11)
        ax.set_ylabel('Accuracy', fontsize=11)
        ax.set_title('Accuracy with Confidence Bands (Defenses vs Fairness Poison)', fontsize=12)
        ax.legend(loc='lower left', fontsize=9)
        ax.grid(True, alpha=0.3)

        # Plot 4: DP for different defenses
        ax = axes[1, 1]

        for defense in defenses:
            key = (attack, defense)
            if key in time_series:
                data = np.array(time_series[key]['dp'])
                mean = np.mean(data, axis=0)
                std = np.std(data, axis=0)

                color = Visualizer.DEFENSE_COLORS.get(defense, 'gray')
                ax.plot(rounds, mean, label=defense, color=color, linewidth=2)
                ax.fill_between(rounds, mean - std, mean + std, color=color, alpha=0.2)

        ax.axvline(x=Config.ATTACK_START_ROUND + 1, color='gray', linestyle='--', alpha=0.7)
        ax.set_xlabel('Round', fontsize=11)
        ax.set_ylabel('Demographic Parity Gap', fontsize=11)
        ax.set_title('DP Gap with Confidence Bands (Defenses vs Fairness Poison)', fontsize=12)
        ax.legend(loc='upper left', fontsize=9)
        ax.grid(True, alpha=0.3)

        plt.tight_layout(rect=[0, 0.03, 1, 0.95])

        # Save figure
        filename = os.path.join(save_dir, 'adult_time_series_comparison.png')
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"[SAVED] {filename}")
        plt.close()

    @staticmethod
    def generate_heatmap(all_runs, seeds, save_dir):
        """Generate heatmap of attack/defense combinations"""

        # Collect mean metrics
        metrics = defaultdict(lambda: {'accuracy': [], 'dp': []})

        for run_results in all_runs:
            for (attack, defense), history in run_results.items():
                key = (attack, defense)
                metrics[key]['accuracy'].append(history['accuracy'][-1])
                metrics[key]['dp'].append(history['demographic_parity'][-1])

        # Get unique attacks and defenses
        attacks = sorted(set(k[0] for k in metrics.keys()))
        defenses = sorted(set(k[1] for k in metrics.keys()))

        # Create matrices
        acc_matrix = np.full((len(attacks), len(defenses)), np.nan)
        dp_matrix = np.full((len(attacks), len(defenses)), np.nan)

        for i, attack in enumerate(attacks):
            for j, defense in enumerate(defenses):
                key = (attack, defense)
                if key in metrics:
                    acc_matrix[i, j] = np.mean(metrics[key]['accuracy'])
                    dp_matrix[i, j] = np.mean(metrics[key]['dp'])

        # Create heatmaps
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))
        fig.suptitle('Adult Dataset - Attack-Defense Performance Heatmaps',
                     fontsize=16, fontweight='bold')

        # Accuracy heatmap
        ax = axes[0]
        im = ax.imshow(acc_matrix, cmap='RdYlGn', aspect='auto', vmin=0.6, vmax=0.9)
        ax.set_xticks(range(len(defenses)))
        ax.set_yticks(range(len(attacks)))
        ax.set_xticklabels(defenses, rotation=45, ha='right')
        ax.set_yticklabels(attacks)
        ax.set_title('Accuracy (higher is better)', fontsize=12)
        ax.set_xlabel('Defense')
        ax.set_ylabel('Attack')
        plt.colorbar(im, ax=ax, shrink=0.8)

        # Add text annotations
        for i in range(len(attacks)):
            for j in range(len(defenses)):
                if not np.isnan(acc_matrix[i, j]):
                    ax.text(j, i, f'{acc_matrix[i, j]:.3f}',
                           ha='center', va='center', fontsize=9, fontweight='bold')

        # DP Gap heatmap
        ax = axes[1]
        im = ax.imshow(dp_matrix, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=0.4)
        ax.set_xticks(range(len(defenses)))
        ax.set_yticks(range(len(attacks)))
        ax.set_xticklabels(defenses, rotation=45, ha='right')
        ax.set_yticklabels(attacks)
        ax.set_title('Demographic Parity Gap (lower is better)', fontsize=12)
        ax.set_xlabel('Defense')
        ax.set_ylabel('Attack')
        plt.colorbar(im, ax=ax, shrink=0.8)

        # Add text annotations
        for i in range(len(attacks)):
            for j in range(len(defenses)):
                if not np.isnan(dp_matrix[i, j]):
                    ax.text(j, i, f'{dp_matrix[i, j]:.3f}',
                           ha='center', va='center', fontsize=9, fontweight='bold')

        plt.tight_layout(rect=[0, 0.03, 1, 0.95])

        filename = os.path.join(save_dir, 'adult_heatmap_attack_defense.png')
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"[SAVED] {filename}")
        plt.close()


# ==============================================================================
# SUMMARY PRINTER
# ==============================================================================

class SummaryPrinter:
    """Class for printing experiment summaries"""

    @staticmethod
    def print_single_seed_summary(results, seed):
        """Print summary for a single seed"""
        print("\n" + "=" * 80)
        print(f"SUMMARY TABLE - ADULT DATASET - SEED {seed}")
        print("=" * 80)
        print(f"{'Attack':<20} {'Defense':<15} {'Accuracy':>10} {'DP Gap':>10} {'EO Gap':>10}")
        print("-" * 80)

        for (attack, defense) in sorted(results.keys()):
            h = results[(attack, defense)]
            print(f"{attack:<20} {defense:<15} {h['accuracy'][-1]:>10.4f} "
                  f"{h['demographic_parity'][-1]:>10.4f} {h['equalized_odds'][-1]:>10.4f}")

        print("=" * 80)

    @staticmethod
    def print_multi_seed_summary(all_runs, seeds):
        """Print aggregate summary across all seeds"""
        print("\n" + "=" * 80)
        print(f"AGGREGATE SUMMARY - ADULT DATASET - SEEDS: {seeds}")
        print("=" * 80)

        # Collect all metrics
        metrics = defaultdict(lambda: {'accuracy': [], 'dp': [], 'eo': []})

        for run_results in all_runs:
            for (attack, defense), history in run_results.items():
                key = (attack, defense)
                metrics[key]['accuracy'].append(history['accuracy'][-1])
                metrics[key]['dp'].append(history['demographic_parity'][-1])
                metrics[key]['eo'].append(history['equalized_odds'][-1])

        print(f"{'Attack':<18} {'Defense':<12} {'Acc (mean±std)':>16} {'DP (mean±std)':>16} {'EO (mean±std)':>16}")
        print("-" * 80)

        for (attack, defense) in sorted(metrics.keys()):
            m = metrics[(attack, defense)]
            acc_mean, acc_std = np.mean(m['accuracy']), np.std(m['accuracy'])
            dp_mean, dp_std = np.mean(m['dp']), np.std(m['dp'])
            eo_mean, eo_std = np.mean(m['eo']), np.std(m['eo'])

            print(f"{attack:<18} {defense:<12} {acc_mean:>7.4f}±{acc_std:<6.4f} "
                  f"{dp_mean:>7.4f}±{dp_std:<6.4f} {eo_mean:>7.4f}±{eo_std:<6.4f}")

        print("=" * 80)

        # Key findings
        print("\n" + "=" * 80)
        print("KEY FINDINGS - ADULT DATASET")
        print("=" * 80)

        # Baseline comparison
        baseline_key = ('none', 'fedavg')
        if baseline_key in metrics:
            baseline_dp = np.mean(metrics[baseline_key]['dp'])
            baseline_acc = np.mean(metrics[baseline_key]['accuracy'])
            print(f"\n1. BASELINE (no attack, fedavg):")
            print(f"   Accuracy: {baseline_acc:.4f}")
            print(f"   DP Gap: {baseline_dp:.4f}")

            print("\n2. IMPACT OF ATTACKS (vs baseline):")
            for attack in ['fairness_poison', 'fairness_spoof', 'adaptive']:
                key = (attack, 'fedavg')
                if key in metrics:
                    attack_dp = np.mean(metrics[key]['dp'])
                    attack_acc = np.mean(metrics[key]['accuracy'])
                    print(f"   {attack}:")
                    print(f"     Accuracy: {attack_acc:.4f} (Δ = {attack_acc - baseline_acc:+.4f})")
                    print(f"     DP Gap: {attack_dp:.4f} (Δ = {attack_dp - baseline_dp:+.4f})")

        # Defense comparison
        print("\n3. DEFENSE EFFECTIVENESS (vs fairness_poison attack):")
        attack = 'fairness_poison'
        baseline_defense = 'fedavg'

        if (attack, baseline_defense) in metrics:
            baseline_dp = np.mean(metrics[(attack, baseline_defense)]['dp'])
            baseline_acc = np.mean(metrics[(attack, baseline_defense)]['accuracy'])

            for defense in ['median', 'robust_fair']:
                key = (attack, defense)
                if key in metrics:
                    defense_dp = np.mean(metrics[key]['dp'])
                    defense_acc = np.mean(metrics[key]['accuracy'])
                    dp_improvement = ((baseline_dp - defense_dp) / baseline_dp) * 100 if baseline_dp > 0 else 0
                    print(f"   {defense}:")
                    print(f"     DP: {defense_dp:.4f} ({dp_improvement:+.1f}% improvement)")
                    print(f"     Acc: {defense_acc:.4f} (Δ = {defense_acc - baseline_acc:+.4f})")

        print("=" * 80)


# ==============================================================================
# MAIN EXECUTION
# ==============================================================================

def run_single_seed(seed, experiments, X, y, sensitive):
    """Run all experiments for a single seed"""
    print(f"\n{'='*80}")
    print(f"RUNNING EXPERIMENTS FOR SEED {seed}")
    print(f"{'='*80}")

    # Set seed and create clients
    Config.set_seed(seed)
    clients = DataLoader.create_clients(X, y, sensitive)
    input_dim = X.shape[1]

    print(f"Clients: {len(clients)}, Adversaries: {Config.ADVERSARY_INDICES}")
    print()

    results = {}

    for idx, (attack, defense) in enumerate(experiments):
        print(f"[{idx+1:2d}/{len(experiments)}] {attack:<20} vs {defense:<15}", end=" ", flush=True)

        try:
            history = ExperimentRunner.run_single_experiment(
                attack, defense, X, y, sensitive, clients, input_dim
            )
            results[(attack, defense)] = history
            print(f"| Acc: {history['accuracy'][-1]:.3f} | DP: {history['demographic_parity'][-1]:.3f}")
        except Exception as e:
            print(f"| ERROR: {str(e)[:40]}")

    return results


def main():
    """Main execution function"""
    print("=" * 80)
    print("EXPERIMENT 2: ADULT INCOME DATASET")
    print("Federated Learning Fairness - Multi-Seed Analysis")
    print("=" * 80)

    # Create output directory
    Config.create_output_dir()

    # Load data once (will be reused across seeds)
    print("\n[STEP 1] Loading Adult Income Dataset...")
    X, y, sensitive = DataLoader.load_adult_data()
    input_dim = X.shape[1]

    # Define experiments
    experiments = [
        # Baselines
        ('none', 'fedavg'),
        ('none', 'median'),
        ('none', 'fairness_only'),
        ('none', 'robust_fair'),

        # Attacks vs FedAvg
        ('random', 'fedavg'),
        ('fairness_poison', 'fedavg'),
        ('label_flip', 'fedavg'),
        ('gradient_scale', 'fedavg'),
        ('fairness_spoof', 'fedavg'),
        ('adaptive', 'fedavg'),

        # Defenses vs fairness_poison
        ('fairness_poison', 'median'),
        ('fairness_poison', 'trimmed'),
        ('fairness_poison', 'krum'),
        ('fairness_poison', 'robust_fair'),

        # Defenses vs other attacks
        ('fairness_spoof', 'robust_fair'),
        ('adaptive', 'robust_fair'),
        ('adaptive', 'median'),
    ]

    # Seeds to run
    seeds = [42, 100, 999]

    print(f"\n[STEP 2] Running {len(experiments)} experiments for each of {len(seeds)} seeds...")

    # Run experiments for each seed
    all_runs = []

    for seed in seeds:
        results = run_single_seed(seed, experiments, X, y, sensitive)
        all_runs.append(results)

        # Print summary for this seed
        SummaryPrinter.print_single_seed_summary(results, seed)

        # Generate plots for this seed
        print(f"\n[PLOTTING] Generating visualizations for seed {seed}...")
        Visualizer.plot_single_seed_results(results, seed, Config.OUTPUT_DIR)

    # Generate comparison plots across all seeds
    print("\n" + "=" * 80)
    print("[STEP 3] GENERATING CROSS-SEED COMPARISON PLOTS")
    print("=" * 80)

    Visualizer.plot_comparison_across_seeds(all_runs, seeds, Config.OUTPUT_DIR)
    Visualizer.plot_time_series_comparison(all_runs, seeds, Config.OUTPUT_DIR)
    Visualizer.generate_heatmap(all_runs, seeds, Config.OUTPUT_DIR)

    # Print aggregate summary
    SummaryPrinter.print_multi_seed_summary(all_runs, seeds)

    print("\n" + "=" * 80)
    print("EXPERIMENT COMPLETE")
    print(f"All results saved to: {Config.OUTPUT_DIR}/")
    print("=" * 80)

    # List all generated files
    print("\nGenerated files:")
    for f in os.listdir(Config.OUTPUT_DIR):
        print(f"  - {f}")

    return all_runs


# ==============================================================================
# RUN
# ==============================================================================

if __name__ == "__main__":
    all_runs = main()

EXPERIMENT 2: ADULT INCOME DATASET
Federated Learning Fairness - Multi-Seed Analysis
[INFO] Created output directory: adult_experiment_results

[STEP 1] Loading Adult Income Dataset...
[DATA] Downloading Adult dataset...
[DATA] Dataset cached to adult_data.csv
[DATA] Adult Dataset loaded: 48842 samples, 13 features
[DATA] Positive rate: 0.239, Male rate: 0.668

[STEP 2] Running 17 experiments for each of 3 seeds...

RUNNING EXPERIMENTS FOR SEED 42
Clients: 6, Adversaries: [4, 5]

[ 1/17] none                 vs fedavg          | Acc: 0.848 | DP: 0.192
[ 2/17] none                 vs median          | Acc: 0.856 | DP: 0.090
[ 3/17] none                 vs fairness_only   | Acc: 0.837 | DP: 0.223
[ 4/17] none                 vs robust_fair     | Acc: 0.844 | DP: 0.214
[ 5/17] random               vs fedavg          | Acc: 0.868 | DP: 0.183
[ 6/17] fairness_poison      vs fedavg          | Acc: 0.797 | DP: 0.024
[ 7/17] label_flip           vs fedavg          | Acc: 0.833 | DP: 0.226
[ 8/